Implementazione della funzione di Attenzione

In [ ]:
# Q, K, V: tensori di forma (batch, seq_len, d_k) o (batch, heads, seq, d_k)
output = F.scaled_dot_product_attention(
    query,           # (B, H, N, d_k)
    key,             # (B, H, M, d_k)
    value,           # (B, H, M, d_v)
    attn_mask=None,  # maschera opzionale booleana o float
    dropout_p=0.0,   # dropout sui pesi di attenzione
    is_causal=False  # applica maschera causale automaticamente
)

Implementazione MultiHead Atteantion

In [ ]:
import torch
import torch.nn as nn

# Inizializzazione del livello
mha = nn.MultiheadAttention(
    embed_dim=512, # d_model
    num_heads=8,   # h
    dropout=0.1,
    batch_first=True # input: (batch, seq, embed_dim)
)

# Forward pass
# query, key, value: (batch, seq_len, embed_dim)
output, attn_weights = mha(
    query=query,
    key=key,
    value=value,
    key_padding_mask=padding_mask, # (batch, seq_len), bool
    attn_mask=causal_mask          # (seq_len, seq_len), bool/float
)

# output: (batch, seq_len, embed_dim)
# attn_weights: (batch, num_heads, seq_len, seq_len) se average_attn_weights=False

Implementazione Positional Encoding

In [ ]:
import torch
import torch.nn as nn
import math

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Matrice PE: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term) # dim pari
        pe[:, 1::2] = torch.cos(position * div_term) # dim dispari

        # Aggiunge dimensione batch: (1, max_len, d_model)
        pe = pe.unsqueeze(0)

        # Registra come buffer (non e' un parametro addestrabile)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len, d_model)
        """
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)